[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/C.%20Core%20Yard%20%26%20Land-Side%20Operations%20%28The%20Heart%20of%20the%20Terminal%29/19.%20The%20SLAP%20for%20Reefer%20Containers%20Problem/P19-Tier-1_executed.ipynb)

---

# 19. The SLAP for Reefer Containers Problem
## Tier 1: Mixed-Integer Programming Formulation

### Purpose and scope

This notebook assigns refrigerated containers (reefers) to storage locations. It formulates the assignment as a mixed-integer optimization problem, solves it with PuLP when PuLP is available, and retains exhaustive enumeration as a fallback for small instances. The inputs are container requirements, location limits and costs, and container-location assignment costs. The outputs are an assignment, location power usage, and a cost breakdown; a visualization and capacity sensitivity analysis help interpret the solution.

The model assumes that each container stays at one location for its full dwell time, its power draw is constant, and its required temperature is represented by a single value. A location's temperature range is inclusive. Power capacity is enforced separately at each location. The model does not represent time-varying loads, uncertainty, movement costs, or interactions between containers other than their combined power use. Costs are treated as additive, except for the stated peak-power penalty.

### Modeling assumptions and units

- Each container must be assigned to exactly one temperature-compatible location.
- A location incurs its fixed activation cost if it serves at least one container.
- Container power is measured in kilowatts (kW), dwell time in hours (h), and the energy price in dollars per kilowatt-hour ($/kWh). Thus, energy use is power multiplied by time, in kWh; no factor of 1,000 is needed.
- The peak-power penalty is a quadratic cost with coefficient `alpha`. It is included in the objective and reported separately from assignment, activation, and energy costs. Since the squared load is measured in kW², `alpha` has units of dollars per kW².
- The MIP expresses this quadratic penalty exactly using binary product variables and linear constraints. This keeps the optimization model compatible with PuLP's linear MIP interface without changing the objective being modeled.

### Solution sequence

1. Define the container and location data.
2. Determine which container-location pairs satisfy the temperature requirement.
3. Formulate assignment, activation, and power variables, the total-cost objective, and the feasibility constraints.
4. Solve the MIP, or enumerate compatible assignments if PuLP is unavailable.
5. Check and interpret the assignment, utilization, cost components, plots, and capacity scenarios.

## Mathematical model

Let $C$ be the set of containers and $L$ the set of storage locations. Let $p_c$ be the power draw (kW), $h_c$ the dwell time (h), and $r_l$ the energy price ($/kWh) for location $l$. Let $a_{cl}$ be the assignment cost, $F_l$ the activation cost, and $K_l$ the power capacity. Define $A \subseteq C \times L$ as the set of temperature-compatible container-location pairs. Only pairs in $A$ can be assigned.

The binary decision variable $x_{cl}$ equals 1 when container $c$ is assigned to location $l$. The binary variable $z_l$ equals 1 when location $l$ is active. The continuous variable $y_l$ is the total power used at location $l$. The binary variable $w_{cdl}$ represents the product $x_{cl}x_{dl}$ for two distinct containers $c<d$ assigned to the same location. The variables have domains $x_{cl},z_l,w_{cdl}\in\{0,1\}$ and $y_l\geq 0$. This product variable lets a linear MIP represent the quadratic power penalty exactly.

The objective minimizes assignment, activation, energy, and peak-power costs:

$$\min \quad \sum_{(c,l)\in A} a_{cl}x_{cl}
+ \sum_{l\in L} F_l z_l
+ \sum_{(c,l)\in A} p_c h_c r_l x_{cl}
+ \alpha \sum_{l\in L} y_l^2.$$

The four terms represent, respectively, the cost of choosing container-location pairs, fixed charges for used locations, electricity consumed over each container's dwell time, and a penalty on high simultaneous power loads. Since kW multiplied by hours gives kWh, the energy-cost term has units of dollars. The coefficient $\alpha$ has units of dollars per kW², so the peak-power term also has units of dollars.

Because $x_{cl}$ is binary and $y_l=\sum_{c:(c,l)\in A}p_cx_{cl}$, the squared term has the exact expansion

$$y_l^2 = \sum_{c:(c,l)\in A}p_c^2x_{cl}
+ 2\sum_{c<d:\,(c,l),(d,l)\in A}p_cp_dw_{cdl}.$$

For each such pair, the linear constraints $w_{cdl}\leq x_{cl}$, $w_{cdl}\leq x_{dl}$, and $w_{cdl}\geq x_{cl}+x_{dl}-1$ enforce $w_{cdl}=x_{cl}x_{dl}$. If either assignment variable is 0, the upper bounds force $w_{cdl}=0$; if both are 1, the lower bound forces it to 1. Thus, the objective remains the stated quadratic cost while the model passed to PuLP is linear.

The constraints are:

$$\sum_{l:(c,l)\in A}x_{cl}=1 \quad \forall c\in C,$$

$$y_l=\sum_{c:(c,l)\in A}p_cx_{cl}, \qquad
y_l\leq K_lz_l \quad \forall l\in L,$$

$$\sum_{c:(c,l)\in A}x_{cl}\leq |C|z_l \quad \forall l\in L.$$

The first constraint assigns every container exactly once. The power equation defines each location's load, and the capacity constraint limits that load. When $z_l=0$, the capacity constraint forces $y_l=0$; when a location has an assignment, the final constraint forces $z_l=1$. Together, these constraints connect assignment decisions to location activation.

In the code, `x`, `z`, and `y` hold the assignment, activation, and power variables, while `w` holds the binary product variables. The coefficient `alpha` scales the squared-power penalty. With $|A|$ compatible assignment pairs, the MIP has a number of assignment variables proportional to $|A|$ and product variables proportional to the number of compatible container pairs at each location. The enumeration fallback considers up to $\prod_{c\in C}|L_c|$ assignments, where $L_c$ is the set of locations compatible with container $c$; this grows rapidly as the instance grows.

In [ ]:
from itertools import combinations, product
from math import isclose
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from dataclasses import dataclass

try:
    import pulp
    PULP_AVAILABLE = True
except ImportError:
    PULP_AVAILABLE = False

print(f"PuLP available: {PULP_AVAILABLE}")

In [ ]:
@dataclass
class ReeferContainer:
    """Container requirements and operating characteristics."""

    id: int
    energy: float  # Constant power draw in kW.
    temp_req: float  # Required temperature in degrees Celsius.
    dwell_time: float  # Time at the storage location in hours.
    cargo_type: str


@dataclass
class StorageLocation:
    """Location capacity, temperature range, and cost parameters."""

    id: int
    power_cap: float  # Maximum power in kW.
    temp_min: float  # Minimum supported temperature in degrees Celsius.
    temp_max: float  # Maximum supported temperature in degrees Celsius.
    activation_cost: float  # Fixed cost incurred when the location is used.
    energy_cost_rate: float  # Energy price in dollars per kWh.


@dataclass
class AssignmentCost:
    """Cost of assigning one container to one location."""

    container_id: int
    location_id: int
    cost: float


print("Data classes defined for reefer container assignment.")

In [ ]:
# Example instance: three containers and two storage locations.
containers = [
    ReeferContainer(
        id=1,
        energy=45,
        temp_req=-18,
        dwell_time=48,
        cargo_type="Pharmaceutical",
    ),
    ReeferContainer(
        id=2,
        energy=35,
        temp_req=2,
        dwell_time=72,
        cargo_type="Fresh Produce",
    ),
    ReeferContainer(
        id=3,
        energy=50,
        temp_req=-20,
        dwell_time=96,
        cargo_type="Frozen Seafood",
    ),
]

locations = [
    StorageLocation(
        id=1,
        power_cap=100,
        temp_min=-25,
        temp_max=5,
        activation_cost=50,
        energy_cost_rate=0.15,
    ),
    StorageLocation(
        id=2,
        power_cap=80,
        temp_min=-15,
        temp_max=10,
        activation_cost=40,
        energy_cost_rate=0.12,
    ),
]

# Assignment costs by (container ID, location ID).
assignment_costs = {
    (1, 1): 15,
    (1, 2): 18,
    (2, 1): 20,
    (2, 2): 12,
    (3, 1): 25,
    (3, 2): 30,
}

print(f"Created {len(containers)} containers and {len(locations)} locations.")
print("\nContainer details:")
for container in containers:
    print(
        f"  Container {container.id}: {container.cargo_type}, "
        f"{container.energy} kW, {container.temp_req}°C, "
        f"{container.dwell_time} h"
    )

print("\nLocation details:")
for location in locations:
    print(
        f"  Location {location.id}: {location.power_cap} kW, "
        f"[{location.temp_min}°C, {location.temp_max}°C], "
        f"activation cost ${location.activation_cost:.2f}, "
        f"energy price ${location.energy_cost_rate:.2f}/kWh"
    )

In [ ]:
def check_temperature_compatibility(
    container: ReeferContainer,
    location: StorageLocation,
) -> bool:
    """Return whether the required temperature lies within the location range."""
    return location.temp_min <= container.temp_req <= location.temp_max


def calculate_total_cost(
    container: ReeferContainer,
    location: StorageLocation,
    assignment_cost: float,
) -> float:
    """Return assignment plus energy cost for one container-location pair."""
    energy_kwh = container.energy * container.dwell_time
    energy_cost = energy_kwh * location.energy_cost_rate
    return assignment_cost + energy_cost


print("Temperature compatibility matrix:")
compatibility_matrix = []
for container in containers:
    row = []
    for location in locations:
        compatible = check_temperature_compatibility(container, location)
        row.append("Compatible" if compatible else "Not compatible")
        print(
            f"  Container {container.id} → Location {location.id}: "
            f"{row[-1]}"
        )
    compatibility_matrix.append(row)

## From the model to the solver

Temperature compatibility determines the feasible assignment pairs before optimization. The solver creates an assignment variable only for each compatible pair, so an incompatible assignment cannot enter a solution. The equality constraints then require exactly one selected pair for each container.

The fixed activation charge is connected to assignments through binary activation variables. Power is the sum of the assigned containers' kW draws, and the capacity constraint limits that sum. Energy cost is based on kWh: a constant draw in kW multiplied by dwell time in hours. The peak-power term penalizes the square of each location's total kW load; its exact binary-product representation is constructed in the MIP function.

In [ ]:
def solve_mip_exact(
    containers: List[ReeferContainer],
    locations: List[StorageLocation],
    assignment_costs: Dict[Tuple[int, int], float],
) -> Dict:
    """Solve the assignment model exactly with a mixed-integer program."""
    if not PULP_AVAILABLE:
        return solve_by_enumeration(containers, locations, assignment_costs)

    compatible_pairs = [
        (container, location)
        for container in containers
        for location in locations
        if check_temperature_compatibility(container, location)
    ]
    compatible_container_ids = {
        container.id for container, _ in compatible_pairs
    }
    if any(container.id not in compatible_container_ids for container in containers):
        return {
            "status": "infeasible",
            "message": "At least one container has no temperature-compatible location.",
        }

    model = pulp.LpProblem("Reefer_Assignment", pulp.LpMinimize)

    # Named arguments make each variable's bounds and category explicit across PuLP versions.
    x = {
        (container.id, location.id): pulp.LpVariable(
            f"x_{container.id}_{location.id}",
            lowBound=None,
            upBound=None,
            cat=pulp.LpBinary,
        )
        for container, location in compatible_pairs
    }
    z = {
        location.id: pulp.LpVariable(
            f"z_{location.id}",
            lowBound=None,
            upBound=None,
            cat=pulp.LpBinary,
        )
        for location in locations
    }
    y = {
        location.id: pulp.LpVariable(
            f"y_{location.id}",
            lowBound=0,
            upBound=None,
            cat=pulp.LpContinuous,
        )
        for location in locations
    }

    # Binary products make the squared load exact while keeping the PuLP model linear.
    w = {}
    for location in locations:
        compatible_containers = [
            container
            for container in containers
            if (container.id, location.id) in x
        ]
        for first, second in combinations(compatible_containers, 2):
            key = (first.id, second.id, location.id)
            w[key] = pulp.LpVariable(
                f"w_{first.id}_{second.id}_{location.id}",
                lowBound=None,
                upBound=None,
                cat=pulp.LpBinary,
            )
            model += w[key] <= x[(first.id, location.id)]
            model += w[key] <= x[(second.id, location.id)]
            model += (
                w[key]
                >= x[(first.id, location.id)]
                + x[(second.id, location.id)]
                - 1
            )

    alpha = 0.001  # Cost coefficient per squared kW of location load.
    objective_terms = []

    for container, location in compatible_pairs:
        key = (container.id, location.id)
        assignment_cost = assignment_costs.get(key, 0)
        energy_cost = (
            container.energy
            * container.dwell_time
            * location.energy_cost_rate
        )
        objective_terms.append((assignment_cost + energy_cost) * x[key])

    for location in locations:
        objective_terms.append(location.activation_cost * z[location.id])

        compatible_containers = [
            container
            for container in containers
            if (container.id, location.id) in x
        ]
        for container in compatible_containers:
            objective_terms.append(
                alpha
                * container.energy**2
                * x[(container.id, location.id)]
            )
        for first, second in combinations(compatible_containers, 2):
            objective_terms.append(
                2
                * alpha
                * first.energy
                * second.energy
                * w[(first.id, second.id, location.id)]
            )

    model += pulp.lpSum(objective_terms)

    # Every container is assigned once; these equalities define each location's load.
    for container in containers:
        model += pulp.lpSum(
            x[(container.id, location.id)]
            for location in locations
            if (container.id, location.id) in x
        ) == 1, f"assign_container_{container.id}"

    for location in locations:
        power_usage = pulp.lpSum(
            container.energy * x[(container.id, location.id)]
            for container in containers
            if (container.id, location.id) in x
        )
        model += y[location.id] == power_usage, f"define_power_{location.id}"
        model += (
            y[location.id] <= location.power_cap * z[location.id],
            f"power_cap_{location.id}",
        )
        model += pulp.lpSum(
            x[(container.id, location.id)]
            for container in containers
            if (container.id, location.id) in x
        ) <= len(containers) * z[location.id], f"activate_link_{location.id}"

    model.solve(pulp.PULP_CBC_CMD(msg=False))
    if model.status != pulp.LpStatusOptimal:
        status_name = pulp.LpStatus[model.status].lower()
        return {
            "status": status_name,
            "message": pulp.LpStatus[model.status],
        }

    container_by_id = {container.id: container for container in containers}
    location_by_id = {location.id: location for location in locations}
    assignments = {}
    location_usage = {
        location.id: {
            "power_used": 0,
            "containers": [],
            "activated": z[location.id].value() > 0.5,
        }
        for location in locations
    }

    for (container_id, location_id), variable in x.items():
        if variable.value() > 0.5:
            assignments[container_id] = location_id
            location_usage[location_id]["power_used"] += (
                container_by_id[container_id].energy
            )
            location_usage[location_id]["containers"].append(container_id)

    assignment_component = sum(
        assignment_costs.get((container_id, location_id), 0)
        for container_id, location_id in assignments.items()
    )
    activation_component = sum(
        location.activation_cost
        for location in locations
        if location_usage[location.id]["activated"]
    )
    energy_component = sum(
        container_by_id[container_id].energy
        * container_by_id[container_id].dwell_time
        * location_by_id[location_id].energy_cost_rate
        for container_id, location_id in assignments.items()
    )
    peak_power_component = alpha * sum(
        usage["power_used"] ** 2 for usage in location_usage.values()
    )

    return {
        "status": "optimal",
        "assignments": assignments,
        "total_cost": pulp.value(model.objective),
        "location_usage": location_usage,
        "objective_breakdown": {
            "assignment_cost": assignment_component,
            "activation_cost": activation_component,
            "energy_cost": energy_component,
            "peak_power_penalty": peak_power_component,
        },
    }


print("MIP solver function defined.")

In [ ]:
def solve_by_enumeration(
    containers: List[ReeferContainer],
    locations: List[StorageLocation],
    assignment_costs: Dict[Tuple[int, int], float],
) -> Dict:
    """Find an exact solution by checking every compatible assignment."""
    compatible_locations = {
        container.id: [
            location.id
            for location in locations
            if check_temperature_compatibility(container, location)
        ]
        for container in containers
    }
    if any(not choices for choices in compatible_locations.values()):
        return {
            "status": "infeasible",
            "message": "At least one container has no temperature-compatible location.",
        }

    container_by_id = {container.id: container for container in containers}
    location_by_id = {location.id: location for location in locations}
    alpha = 0.001
    best_solution = None
    best_cost = float("inf")
    best_breakdown = None

    # Exhaustive enumeration is practical only when the product of choices is small.
    choice_lists = [compatible_locations[container.id] for container in containers]
    for assignment_tuple in product(*choice_lists):
        assignments = {
            container.id: assignment_tuple[index]
            for index, container in enumerate(containers)
        }
        location_power = {location.id: 0 for location in locations}
        for container_id, location_id in assignments.items():
            location_power[location_id] += container_by_id[container_id].energy

        if any(
            location_power[location.id] > location.power_cap
            for location in locations
        ):
            continue

        assignment_component = sum(
            assignment_costs.get((container_id, location_id), 0)
            for container_id, location_id in assignments.items()
        )
        activated_locations = set(assignments.values())
        activation_component = sum(
            location_by_id[location_id].activation_cost
            for location_id in activated_locations
        )
        energy_component = sum(
            calculate_total_cost(
                container_by_id[container_id],
                location_by_id[location_id],
                0,
            )
            for container_id, location_id in assignments.items()
        )
        peak_power_component = alpha * sum(
            power**2 for power in location_power.values()
        )
        total_cost = (
            assignment_component
            + activation_component
            + energy_component
            + peak_power_component
        )

        # Keep the first solution when objective values tie within floating-point tolerance.
        if total_cost < best_cost and not isclose(
            total_cost, best_cost, rel_tol=1e-9, abs_tol=1e-9
        ):
            best_cost = total_cost
            best_solution = assignments
            best_breakdown = {
                "assignment_cost": assignment_component,
                "activation_cost": activation_component,
                "energy_cost": energy_component,
                "peak_power_penalty": peak_power_component,
            }

    if best_solution is None:
        return {"status": "infeasible", "message": "No feasible assignment exists."}

    location_usage = {
        location.id: {
            "power_used": 0,
            "containers": [],
            "activated": False,
        }
        for location in locations
    }
    for container_id, location_id in best_solution.items():
        location_usage[location_id]["power_used"] += (
            container_by_id[container_id].energy
        )
        location_usage[location_id]["containers"].append(container_id)
        location_usage[location_id]["activated"] = True

    return {
        "status": "optimal",
        "assignments": best_solution,
        "total_cost": best_cost,
        "location_usage": location_usage,
        "objective_breakdown": best_breakdown,
    }


print("Enumeration fallback function defined.")

## Solving and interpreting the assignment

The solver returns an assignment only when it finds a feasible optimum. A feasible result must assign every container once, use only temperature-compatible locations, and respect each location's power capacity. The objective breakdown separates the four modeled cost components; their sum should agree with the reported objective, up to numerical solver tolerance.

A location's utilization is its assigned power divided by its capacity. This percentage describes the share of available kW capacity used; it is not an energy percentage. The peak-power penalty is based on squared kW load and therefore is not an energy charge.

In [ ]:
solution = solve_mip_exact(containers, locations, assignment_costs)

print("REEFER ASSIGNMENT OPTIMIZATION RESULTS")
print(f"Solution status: {solution['status'].upper()}")

if solution["status"] == "optimal":
    container_by_id = {container.id: container for container in containers}
    location_by_id = {location.id: location for location in locations}

    print(f"Total objective cost: ${solution['total_cost']:.2f}")
    print("\nAssignments:")
    for container_id, location_id in solution["assignments"].items():
        container = container_by_id[container_id]
        location = location_by_id[location_id]
        assignment_cost = assignment_costs.get((container_id, location_id), 0)
        pair_cost = calculate_total_cost(
            container, location, assignment_cost
        )
        energy_cost = pair_cost - assignment_cost
        assert check_temperature_compatibility(container, location)
        print(
            f"  Container {container_id} ({container.cargo_type}) → "
            f"Location {location_id}"
        )
        print(
            f"    Required temperature: {container.temp_req}°C; "
            f"location range: [{location.temp_min}°C, {location.temp_max}°C]"
        )
        print(
            f"    Power: {container.energy} kW; "
            f"assignment cost: ${assignment_cost:.2f}; "
            f"energy cost: ${energy_cost:.2f}"
        )

    print("\nLocation utilization:")
    for location in locations:
        usage = solution["location_usage"][location.id]
        utilization = 100 * usage["power_used"] / location.power_cap
        assert usage["power_used"] <= location.power_cap
        print(
            f"  Location {location.id}: {usage['power_used']} / "
            f"{location.power_cap} kW ({utilization:.1f}% utilized)"
        )
        print(f"    Containers: {usage['containers']}")
        print(f"    Status: {'Active' if usage['activated'] else 'Inactive'}")

    breakdown = solution["objective_breakdown"]
    breakdown_total = sum(breakdown.values())
    assert isclose(
        breakdown_total,
        solution["total_cost"],
        rel_tol=1e-7,
        abs_tol=1e-7,
    )
    print("\nObjective breakdown:")
    for component, value in breakdown.items():
        print(f"  {component.replace('_', ' ').title()}: ${value:.2f}")
    print(f"  Component sum: ${breakdown_total:.2f}")
else:
    print(f"No optimal assignment: {solution.get('message', 'No further details.')}")

## Visualizing the solution

The four panels show the selected container-location pairs, power use against capacity, the objective cost components, and temperature compatibility. In the temperature panel, a marker identifies a container's required temperature at a location; the vertical interval shows that location's supported range. A point inside the interval is temperature-compatible. Marker shape and color distinguish selected assignments from other compatible options and incompatible pairs.

In [ ]:
if solution["status"] == "optimal":
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle(
        "Reefer Container Assignment Optimization Results",
        fontsize=16,
        fontweight="bold",
    )

    # The heatmap displays the binary assignment decision for every pair.
    ax1 = axes[0, 0]
    assignment_matrix = np.zeros((len(containers), len(locations)))
    container_index = {container.id: index for index, container in enumerate(containers)}
    location_index = {location.id: index for index, location in enumerate(locations)}
    for container_id, location_id in solution["assignments"].items():
        assignment_matrix[
            container_index[container_id], location_index[location_id]
        ] = 1

    container_labels = [
        f"C{container.id}\n{container.cargo_type}\n{container.energy} kW"
        for container in containers
    ]
    location_labels = [
        f"L{location.id}\n{location.power_cap} kW"
        for location in locations
    ]
    sns.heatmap(
        assignment_matrix,
        annot=True,
        cmap="Blues",
        xticklabels=location_labels,
        yticklabels=container_labels,
        cbar_kws={"label": "Assignment decision (1 = assigned)"},
        ax=ax1,
    )
    ax1.set_title("Container Assignment Matrix")
    ax1.set_xlabel("Storage locations")
    ax1.set_ylabel("Reefer containers")

    # Comparing used power with capacity makes the binding resource visible.
    ax2 = axes[0, 1]
    location_names = [f"Location {location.id}" for location in locations]
    power_used = [
        solution["location_usage"][location.id]["power_used"]
        for location in locations
    ]
    power_capacity = [location.power_cap for location in locations]
    positions = np.arange(len(location_names))
    width = 0.35
    ax2.bar(
        positions - width / 2,
        power_used,
        width,
        label="Power used",
        color="skyblue",
        alpha=0.85,
    )
    ax2.bar(
        positions + width / 2,
        power_capacity,
        width,
        label="Power capacity",
        color="lightcoral",
        alpha=0.85,
    )
    ax2.set_xlabel("Storage locations")
    ax2.set_ylabel("Power (kW)")
    ax2.set_title("Power Utilization by Location")
    ax2.set_xticks(positions)
    ax2.set_xticklabels(location_names)
    ax2.legend()
    ax2.grid(True, alpha=0.3, axis="y")
    for index, (used, capacity) in enumerate(zip(power_used, power_capacity)):
        ax2.text(
            index - width / 2,
            used,
            f"{100 * used / capacity:.1f}%",
            ha="center",
            va="bottom",
            fontsize=9,
        )

    # The objective components sum to the reported total objective value.
    ax3 = axes[1, 0]
    breakdown = solution["objective_breakdown"]
    cost_labels = [
        "Assignment",
        "Activation",
        "Energy",
        "Peak-power penalty",
    ]
    costs = [
        breakdown["assignment_cost"],
        breakdown["activation_cost"],
        breakdown["energy_cost"],
        breakdown["peak_power_penalty"],
    ]
    if sum(costs) > 0:
        ax3.pie(
            costs,
            labels=cost_labels,
            autopct="%1.1f%%",
            startangle=90,
            textprops={"fontsize": 9},
        )
    else:
        ax3.text(0.5, 0.5, "All cost components are zero", ha="center", va="center")
    ax3.set_title("Objective Cost Breakdown")

    # Each location's temperature interval is drawn once; markers show each pair.
    ax4 = axes[1, 1]
    for location_position, location in enumerate(locations):
        ax4.vlines(
            location_position,
            location.temp_min,
            location.temp_max,
            color="steelblue",
            linewidth=5,
            alpha=0.35,
        )
        ax4.hlines(
            [location.temp_min, location.temp_max],
            location_position - 0.12,
            location_position + 0.12,
            color="steelblue",
            linewidth=2,
        )

    for container_position, container in enumerate(containers):
        for location_position, location in enumerate(locations):
            compatible = check_temperature_compatibility(container, location)
            assigned = solution["assignments"].get(container.id) == location.id
            if assigned:
                color, marker = "crimson", "o"
            elif compatible:
                color, marker = "darkgreen", "s"
            else:
                color, marker = "gray", "x"
            ax4.scatter(
                location_position,
                container.temp_req,
                color=color,
                marker=marker,
                s=90,
                edgecolors="black" if marker != "x" else None,
                linewidth=1,
                zorder=5,
            )
            if location_position == 0:
                ax4.annotate(
                    f"C{container.id}",
                    (location_position, container.temp_req),
                    xytext=(-14, 5 + container_position * 2),
                    textcoords="offset points",
                    fontsize=8,
                )

    from matplotlib.lines import Line2D

    legend_elements = [
        Line2D(
            [0], [0], marker="o", color="w", markerfacecolor="crimson",
            markeredgecolor="black", label="Assigned pair", markersize=8,
        ),
        Line2D(
            [0], [0], marker="s", color="w", markerfacecolor="darkgreen",
            markeredgecolor="black", label="Compatible, not assigned", markersize=8,
        ),
        Line2D(
            [0], [0], marker="x", color="gray", label="Incompatible pair",
            markersize=8,
        ),
        Line2D([0], [0], color="steelblue", linewidth=4, label="Location range"),
    ]
    ax4.set_xlabel("Storage locations")
    ax4.set_ylabel("Temperature (°C)")
    ax4.set_title("Temperature Compatibility and Assignment")
    ax4.set_xticks(range(len(locations)))
    ax4.set_xticklabels([f"L{location.id}" for location in locations])
    ax4.grid(True, alpha=0.3)
    ax4.legend(handles=legend_elements, loc="best", fontsize=8)

    plt.tight_layout()
    plt.show()
else:
    print("No solution plots are available because the instance is infeasible.")

## Capacity sensitivity analysis

Sensitivity analysis asks how the optimal result changes when a model parameter changes. Here, each scenario scales every location's power capacity while leaving container data, temperature ranges, costs, and the objective unchanged. Each scenario is solved independently using the same MIP formulation and solution procedure.

A capacity reduction can make the assignment infeasible. If a scenario is feasible, increasing capacity cannot remove any previously feasible assignment, so the optimal objective cannot become worse solely because capacity increased. A low total capacity is one reason an instance can be infeasible, but compatibility and the distribution of power demand across locations also matter: total capacity alone does not establish feasibility.

In [ ]:
print("CAPACITY SENSITIVITY ANALYSIS")

base_capacity = [location.power_cap for location in locations]
scenarios = [
    {"name": "Base case", "capacity_multiplier": 1.0},
    {"name": "Reduced capacity (-20%)", "capacity_multiplier": 0.8},
    {"name": "Increased capacity (+20%)", "capacity_multiplier": 1.2},
    {"name": "Limited capacity (-40%)", "capacity_multiplier": 0.6},
]

scenario_results = []
for scenario in scenarios:
    modified_locations = [
        StorageLocation(
            id=location.id,
            power_cap=base_capacity[index] * scenario["capacity_multiplier"],
            temp_min=location.temp_min,
            temp_max=location.temp_max,
            activation_cost=location.activation_cost,
            energy_cost_rate=location.energy_cost_rate,
        )
        for index, location in enumerate(locations)
    ]

    result = solve_mip_exact(containers, modified_locations, assignment_costs)
    feasible = result["status"] == "optimal"
    total_capacity = sum(location.power_cap for location in modified_locations)
    scenario_results.append(
        {
            "scenario": scenario["name"],
            "total_cost": result["total_cost"] if feasible else float("nan"),
            "feasible": feasible,
            "total_capacity": total_capacity,
        }
    )

    if feasible:
        print(
            f"  {scenario['name']}: objective = "
            f"${result['total_cost']:.2f}; feasible"
        )
    else:
        print(f"  {scenario['name']}: infeasible ({result.get('message', 'No feasible assignment')})")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

feasible_results = [result for result in scenario_results if result["feasible"]]
if feasible_results:
    capacities = [result["total_capacity"] for result in feasible_results]
    costs = [result["total_cost"] for result in feasible_results]
    ax1.plot(capacities, costs, "bo-", linewidth=2, markersize=8)
    ax1.set_xlabel("Total power capacity (kW)")
    ax1.set_ylabel("Optimal objective cost ($)")
    ax1.set_title("Optimal Cost Across Feasible Capacity Scenarios")
    ax1.grid(True, alpha=0.3)
    for capacity, cost, result in zip(capacities, costs, feasible_results):
        ax1.annotate(
            result["scenario"],
            (capacity, cost),
            textcoords="offset points",
            xytext=(0, 8),
            ha="center",
            fontsize=8,
        )
else:
    ax1.text(0.5, 0.5, "No feasible scenarios", ha="center", va="center")
    ax1.set_title("Optimal Cost Across Feasible Capacity Scenarios")

scenario_positions = np.arange(len(scenario_results))
capacity_values = [result["total_capacity"] for result in scenario_results]
bar_colors = [
    "lightgreen" if result["feasible"] else "lightcoral"
    for result in scenario_results
]
bars = ax2.bar(scenario_positions, capacity_values, color=bar_colors, alpha=0.8)
ax2.set_xlabel("Capacity scenario")
ax2.set_ylabel("Total power capacity (kW)")
ax2.set_title("Scenario Feasibility and Total Capacity")
ax2.set_xticks(scenario_positions)
ax2.set_xticklabels(
    [result["scenario"].replace(" ", "\n") for result in scenario_results],
    rotation=25,
    ha="right",
)
ax2.grid(True, alpha=0.3, axis="y")
for bar, result in zip(bars, scenario_results):
    label = (
        f"Feasible\n${result['total_cost']:.2f}"
        if result["feasible"]
        else "Infeasible"
    )
    ax2.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        label,
        ha="center",
        va="bottom",
        fontsize=8,
    )

plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("- Feasibility depends on compatible assignments as well as available power.")
print("- A feasible solution under a capacity limit remains feasible if capacities increase.")
print("- Capacity changes can affect the optimal objective by changing the available assignments.")

## Role, strengths, and limitations of Tier 1

This tier provides an exact optimization formulation for the stated assignment model. When a solver reports an optimal solution, that solution minimizes the modeled objective over all feasible assignments. The enumeration fallback has the same objective and constraints, but its number of candidate assignments grows as the product of the number of compatible choices for each container; it is intended for small instances.

The formulation handles temperature compatibility, assignment, activation, power capacity, energy charges, and the quadratic peak-power penalty together. Its mathematical structure makes the assumptions and cost trade-offs explicit and supports controlled what-if analysis. Exact optimization can require substantial computation as instance size grows, and practical performance depends on the solver and instance structure. The model's conclusions apply only to the inputs and assumptions stated here; real terminal planning may require additional operational constraints and data.